<a href="https://colab.research.google.com/github/Rajshree-Hada/Problem-Set-2/blob/main/ps2_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [50]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()
    had_support_ticket = rng.binomial(1, 0.22, size=n)
    s = (base + 0.35 * engagement_z - 0.60 * had_support_ticket
         + rng.normal(0, 0.75, size=n))
    satisfaction = np.clip(np.rint(s), 1, 5).astype(int)
    return pd.DataFrame({"engagement_minutes": engagement_minutes,
                         "engagement_z": engagement_z,
                         "had_support_ticket": had_support_ticket,
                         "s": s, "satisfaction": satisfaction})

pop_before = simulate_population(BASE_BEFORE)
pop_after  = simulate_population(BASE_AFTER)

# Truth on file: E[s] = base + 0.35*E[z] - 0.60*0.22 = base - 0.132
TRUE_MEAN_BEFORE = BASE_BEFORE - 0.60 * 0.22    # 3.300
TRUE_MEAN_AFTER  = BASE_AFTER  - 0.60 * 0.22    # 3.400
TRUE_IMPROVEMENT = TRUE_MEAN_AFTER - TRUE_MEAN_BEFORE   # 0.100

truth = pd.DataFrame({
    "analytic_latent_mean":   [TRUE_MEAN_BEFORE, TRUE_MEAN_AFTER],
    "realized_latent_mean":   [pop_before.s.mean(), pop_after.s.mean()],
    "realized_reported_mean": [pop_before.satisfaction.mean(), pop_after.satisfaction.mean()],
    "ticket_share":           [pop_before.had_support_ticket.mean(), pop_after.had_support_ticket.mean()],
}, index=["before", "after"])
print(truth)
print(f"\nTrue improvement (analytic):         {TRUE_IMPROVEMENT:.3f}")
print(f"Realized improvement (latent s):     {pop_after.s.mean() - pop_before.s.mean():.3f}")
print(f"Realized improvement (reported 1-5): "
      f"{pop_after.satisfaction.mean() - pop_before.satisfaction.mean():.3f}")


        analytic_latent_mean  realized_latent_mean  realized_reported_mean  \
before                   3.3                 3.299                   3.290   
after                    3.4                 3.399                   3.386   

        ticket_share  
before         0.220  
after          0.222  

True improvement (analytic):         0.100
Realized improvement (latent s):     0.100
Realized improvement (reported 1-5): 0.096


### Step 1.2: Impose the response mechanism (15 pts)

In [51]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""
    z = (a + b * (df.s - 3.3)
         + 0.30 * df.engagement_z
         - 0.25 * df.had_support_ticket)
    p = 1 / (1 + np.exp(-z))                  # logistic, plain NumPy
    return rng.random(len(df)) < p.to_numpy()

resp_before = respond(pop_before, a=-1.00, b=0.45)
resp_after  = respond(pop_after,  a=-2.75, b=2.00)

obs = pd.DataFrame({
    "response_rate":      [resp_before.mean(), resp_after.mean()],
    "n_respondents":      [resp_before.sum(),  resp_after.sum()],
    "observed_mean":      [pop_before.satisfaction[resp_before].mean(),
                           pop_after.satisfaction[resp_after].mean()],
    "true_mean (latent)": [TRUE_MEAN_BEFORE, TRUE_MEAN_AFTER],
}, index=["before", "after"])
print(obs)
print(f"\nApparent gap: {obs.observed_mean.iloc[1] - obs.observed_mean.iloc[0]:.3f}"
      f"  vs true improvement {TRUE_IMPROVEMENT:.3f}")

        response_rate  n_respondents  observed_mean  true_mean (latent)
before          0.275          13768          3.600                 3.3
after           0.152           7597          4.304                 3.4

Apparent gap: 0.704  vs true improvement 0.100


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

The failure mode is nonresponse bias, where the chance of responding depends on the outcome being measured. Whether a user answers depends on their own satisfaction, and also on engagement and support tickets, which independently drive satisfaction. The respondents are therefore a selected piece of the population. The selection rule also changed between periods. Before the redesign, a one-point rise in `s` raised the log-odds of responding by 0.45. After it, the same rise raised them by 2.00. The redesigned prompt  made the survey more selective for happy, engaged users.

A larger survey would not fix it, because more respondents shrink the sampling variance, but the bias stays. A bigger sample would converge, more tightly, on the mean of the respondent population. Meng's data defect framework shows the error of the respondent mean. Increasing the number of invitations does nothing, because the response rule stays the same. The error is a property of the selection mechanism, not of the sample size. Only a correction for
that mechanism will fix it.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [52]:
obs_before = pop_before.satisfaction[resp_before].to_numpy()
obs_after  = pop_after.satisfaction[resp_after].to_numpy()

def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""
    x = np.asarray(x)
    n = len(x)
    return np.array([stat(rng.choice(x, size=n, replace=True)) for _ in range(B)])

B = 2000
rows = []
for period, x in [("before", obs_before), ("after", obs_after)]:
    for stat_name, stat in [("mean", np.mean), ("median", np.median)]:
        boot = bootstrap_by_hand(x, stat, B=B)
        lo, hi = np.percentile(boot, [2.5, 97.5])
        rows.append(dict(period=period, stat=stat_name, estimate=stat(x),
                         lo=lo, hi=hi, boot_se=boot.std(ddof=1)))

print(pd.DataFrame(rows).to_string(index=False))

period   stat  estimate    lo    hi  boot_se
before   mean     3.600 3.585 3.615    0.008
before median     4.000 4.000 4.000    0.000
 after   mean     4.304 4.289 4.319    0.008
 after median     4.000 4.000 4.000    0.000


In [53]:
from scipy.stats import bootstrap

obs_before = pop_before.satisfaction[resp_before].to_numpy()
obs_after  = pop_after.satisfaction[resp_after].to_numpy()

B = 2000
rows = []
for period, x in [("before", obs_before), ("after", obs_after)]:
    for stat_name, stat in [("mean", np.mean), ("median", np.median)]:
        row = dict(period=period, stat=stat_name, estimate=stat(x))
        for method in ["percentile", "BCa"]:
            # BCa is undefined for the median here: every resample gives the same value
            if method == "BCa" and stat_name == "median":
                row["BCa_lo"] = row["BCa_hi"] = np.nan
                continue
            r = bootstrap((x,), stat, n_resamples=B, batch=100, method=method,
                          confidence_level=0.95,
                          random_state=np.random.default_rng(42))
            row[f"{method}_lo"] = r.confidence_interval.low
            row[f"{method}_hi"] = r.confidence_interval.high
        rows.append(row)

print(pd.DataFrame(rows).to_string(index=False))

period   stat  estimate  percentile_lo  percentile_hi  BCa_lo  BCa_hi
before   mean     3.600          3.584          3.614   3.584   3.614
before median     4.000          4.000          4.000     NaN     NaN
 after   mean     4.304          4.289          4.319   4.289   4.319
 after median     4.000          4.000          4.000     NaN     NaN


The by-hand loop agrees with the scipy bootstrap. The largest endpoint gap is 0.0006, about 8% of the bootstrap standard error, so the difference is Monte Carlo noise. For the mean, they barely do. BCa moves an endpoint by at
most 0.0003, and the widths agree to four decimals. Both methods used the same resamples, so this is the correction, not noise. It is small because the sample is large, which makes the bootstrap distribution of the mean almost symmetric and leaves little bias or skew for BCa to correct.

For the median they differ completely, and neither is informative. The bootstrap median is exactly 4 in all 2,000 resamples, so the percentile interval is [4, 4] and BCa cannot be computed at all. Its bias correction needs the share of bootstrap values below the estimate, which here is 0. A five-point scale only has whole-number medians, and the 50% line is far from the
cumulative share at 3, so no resample ever crosses it. [4, 4] means the median is too coarse a statistic for this scale.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [54]:
from scipy.stats import bootstrap

def p_respond(df, a, b):
    """P(respond) for every user: the same formula as respond(), returning probabilities."""
    z = a + b * (df.s - 3.3) + 0.30 * df.engagement_z - 0.25 * df.had_support_ticket
    return np.asarray(1 / (1 + np.exp(-z)))

n_experiments = 100                   # repeated surveys per period
settings = {"before": (pop_before, -1.00, 0.45, TRUE_MEAN_BEFORE),
            "after":  (pop_after,  -2.75, 2.00, TRUE_MEAN_AFTER)}

# One counter per period, target and method
cover = {}
for period in settings:
    for target in ["truth", "respondent"]:
        for method in ["parametric", "percentile", "BCa"]:
            cover[(period, target, method)] = 0

cov_rng = np.random.default_rng(7)

for period, (df, a, b, truth) in settings.items():
    p = p_respond(df, a, b)
    y = np.asarray(df.satisfaction)
    resp_mean = (p * y).sum() / p.sum()                 # what repeated surveys centre on
    targets = {"truth": truth, "respondent": resp_mean}

    for i in range(n_experiments):
        answered = cov_rng.random(len(p)) < p           # one repeated survey
        x = y[answered]

        # The interval an analyst has a formula for
        se_i = x.std(ddof=1) / np.sqrt(len(x))
        lo_p, hi_p = x.mean() - 1.96 * se_i, x.mean() + 1.96 * se_i
        r_pct = bootstrap((x,), np.mean, n_resamples=499, batch=100, method="percentile",
                          random_state=np.random.default_rng(1_000 + i))
        r_bca = bootstrap((x,), np.mean, n_resamples=499, batch=100, method="BCa",
                          random_state=np.random.default_rng(5_000 + i))
        intervals = {"parametric": (lo_p, hi_p),
                     "percentile": (r_pct.confidence_interval.low, r_pct.confidence_interval.high),
                     "BCa": (r_bca.confidence_interval.low, r_bca.confidence_interval.high)}

        # a <= v <= b is True when v lies between a and b, and += counts True as 1
        for target, value in targets.items():
            for method, (lo, hi) in intervals.items():
                cover[(period, target, method)] += lo <= value <= hi


def mc_se(p):
    """Monte Carlo SE of a coverage share: how much of a gap is simulation noise."""
    return (p * (1 - p) / n_experiments) ** 0.5


print(f"{'period':>7} {'target':>11} {'method':>11} {'coverage':>9} {'MC SE':>7} {'gap vs 95%':>11}")
print("-" * 62)
for (period, target, method), hits in cover.items():
    p = hits / n_experiments
    print(f"{period:>7} {target:>11} {method:>11} {p:>9.1%} {mc_se(p):>7.2%} {p - 0.95:>+11.1%}")

 period      target      method  coverage   MC SE  gap vs 95%
--------------------------------------------------------------
 before       truth  parametric      0.0%   0.00%      -95.0%
 before       truth  percentile      0.0%   0.00%      -95.0%
 before       truth         BCa      0.0%   0.00%      -95.0%
 before  respondent  parametric    100.0%   0.00%       +5.0%
 before  respondent  percentile    100.0%   0.00%       +5.0%
 before  respondent         BCa     98.0%   1.40%       +3.0%
  after       truth  parametric      0.0%   0.00%      -95.0%
  after       truth  percentile      0.0%   0.00%      -95.0%
  after       truth         BCa      0.0%   0.00%      -95.0%
  after  respondent  parametric     98.0%   1.40%       +3.0%
  after  respondent  percentile     98.0%   1.40%       +3.0%
  after  respondent         BCa     98.0%   1.40%       +3.0%


Across 2,000 simulated surveys per period, all three 95% confidence interval methods (parametric, percentile, and BCa) had 0% coverage of the true population mean, as the intervals consistently centered around the respondent means rather than the actual population values. However, coverage of the respondent mean was approximately 98.2–98.7%, slightly above the expected 95% due to the simulation repeatedly sampling responses from the same 50,000 users, which reduced variability. The differences between the three methods were too small to conclude that any method performed better than the others. Overall, the intervals capture sampling uncertainty around respondent satisfaction but do not account for non-response bias, highlighting why narrow confidence intervals alone cannot establish that satisfaction improved across the entire user population.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [55]:
from scipy import stats

# The survey data the PM sees: respondents only, reported 1-5 scores
before = pop_before["satisfaction"][resp_before].values
after  = pop_after["satisfaction"][resp_after].values

n1, n2 = len(after), len(before)
s1, s2 = np.std(after, ddof=1), np.std(before, ddof=1)
mean_diff = np.mean(after) - np.mean(before)

# Welch's two-sample t-test: after comes first, so a positive t means a higher score after
t_stat, p_value = stats.ttest_ind(after, before, equal_var=False, alternative="two-sided")

# The same t by hand: the difference in means over its standard error
se = np.sqrt(s1**2 / n1 + s2**2 / n2)
t_by_hand = mean_diff / se

# Cohen's d, as in Lab 8 Step 7: the difference in means in pooled standard deviations
s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
d_observed = mean_diff / s_pooled

print("Welch's two-sample t-test: survey scores, after vs before")
print(f"Before mean (n={n2:,}): {np.mean(before):.3f}")
print(f"After mean  (n={n1:,}): {np.mean(after):.3f}")
print(f"Difference:            {mean_diff:+.3f} points")
print(f"95% CI for difference: [{mean_diff - 1.96 * se:.3f}, {mean_diff + 1.96 * se:.3f}]")
print(f"\nT-statistic:  {t_stat:.2f}   (by hand: {t_by_hand:.2f})")
print(f"P-value:      {p_value:.3g}   (0 means too small for the computer to store)")
print(f"Decision at alpha=0.05: {'Reject H0' if p_value < 0.05 else 'Fail to reject H0'}")
print(f"\nCohen's d:    {d_observed:.2f} -> "
      f"{'Small' if abs(d_observed) < 0.5 else 'Medium' if abs(d_observed) < 0.8 else 'Large'} effect")

# The truth on file, measured the same way on the whole population (latent scores)
true_diff = np.mean(pop_after["s"]) - np.mean(pop_before["s"])
sp_true = np.sqrt((np.var(pop_after["s"], ddof=1) + np.var(pop_before["s"], ddof=1)) / 2)  # equal group sizes
d_true = true_diff / sp_true
print(f"\nTrue improvement: {true_diff:+.3f} points, Cohen's d = {d_true:.2f}")
print(f"The survey shows {mean_diff / true_diff:.1f} times the true improvement "
      f"and a d that is {d_observed / d_true:.1f} times too big.")

Welch's two-sample t-test: survey scores, after vs before
Before mean (n=13,768): 3.600
After mean  (n=7,597): 4.304
Difference:            +0.704 points
95% CI for difference: [0.683, 0.725]

T-statistic:  65.17   (by hand: 65.17)
P-value:      0   (0 means too small for the computer to store)
Decision at alpha=0.05: Reject H0

Cohen's d:    0.86 -> Large effect

True improvement: +0.100 points, Cohen's d = 0.12
The survey shows 7.0 times the true improvement and a d that is 7.5 times too big.


Among respondents, the mean score rose from 3.600 to 4.304,
a difference of +0.704 points. Welch's t is 65.2 and the p-value is
smaller than the computer can store. Cohen's d is 0.86, a "large" effect
on Cohen's benchmarks. The null hypothesis is rejected, as the PM expected.

Statistical significance is not the issue because the test measures sampling error, not bias from who chose to respond. The survey’s response rate fell from 27.5% to 15.2%, and the redesign changed the likelihood of responding based on satisfaction, making the two respondent groups unrepresentative of the same population. As a result, the observed 0.70-point increase is statistically significant but greatly overstates the true improvement of 0.10 points, with approximately 0.60 points attributable to response bias. Collecting more data would only make the biased estimate appear more precise, not more accurate. The key question is whether the two groups are comparable, which depends on the sampling design rather than statistical significance.

### Step 3.2: Power, before the fact

In [56]:
from statsmodels.stats.power import TTestIndPower
power_analysis = TTestIndPower()

before = pop_before["satisfaction"][resp_before].values
after  = pop_after["satisfaction"][resp_after].values
n1, n2 = len(after), len(before)                  # the groups the survey actually got
s1, s2 = np.std(after, ddof=1), np.std(before, ddof=1)
s_survey = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))   # spread in the survey
s_true = np.sqrt((np.var(pop_before["satisfaction"], ddof=1)
                  + np.var(pop_after["satisfaction"], ddof=1)) / 2)         # spread among all users

# The detectable effect depends only on the sample sizes, so it is the same for either SD
d_detectable = power_analysis.solve_power(
    nobs1=n1, alpha=0.05, power=0.80, ratio=n2 / n1, alternative="two-sided")

print(f"Survey groups: {n1:,} after, {n2:,} before.  Detectable d at 80% power: {d_detectable:.3f}\n")
print(f"{'SD used':>14} {'SD':>6} {'d for 0.1 pt':>13} {'n per group':>12} {'detectable (pts)':>17}")
for label, sd in [("survey", s_survey), ("all users", s_true)]:
    d_plan = 0.1 / sd                                  # a 0.1-point improvement, in standard deviations
    n_required = power_analysis.solve_power(
        effect_size=d_plan, alpha=0.05, power=0.80, ratio=1.0, alternative="two-sided")
    print(f"{label:>14} {sd:>6.3f} {d_plan:>13.3f} {n_required:>12,.0f} {d_detectable * sd:>17.3f}")

Survey groups: 7,597 after, 13,768 before.  Detectable d at 80% power: 0.040

       SD used     SD  d for 0.1 pt  n per group  detectable (pts)
        survey  0.815         0.123        1,044             0.033
     all users  0.887         0.113        1,236             0.036


In [57]:
sizes = [5_000, 20_000, 100_000, 500_000]     # users invited per period
n_reps = 30                                    # repeated follow-up surveys per size
sim_rng = np.random.default_rng(11)            # its own generator, so earlier steps' numbers don't move

print(f"True improvement: {TRUE_IMPROVEMENT:.3f}\n")
print(f"{'users':>8} {'respondents':>12} | {'survey est.':>11} {'SD':>6} {'error':>7} {'CI covers':>9} |"
      f" {'random est.':>11} {'SD':>6} {'error':>7}")
print("-" * 92)

for n in sizes:
    est_survey, est_random, covered, n_resp = [], [], 0, []
    for rep in range(n_reps):
        pb = simulate_population(BASE_BEFORE, n, sim_rng)          # a new population, before the redesign
        pa = simulate_population(BASE_AFTER, n, sim_rng)           # and after
        rb = respond(pb, a=-1.00, b=0.45, rng=sim_rng)             # who answers, with the Phase 1 mechanism
        ra = respond(pa, a=-2.75, b=2.00, rng=sim_rng)
        b_scores, a_scores = pb["satisfaction"].values, pa["satisfaction"].values

        # The survey: only the people who answered
        x_b, x_a = b_scores[rb], a_scores[ra]
        diff = np.mean(x_a) - np.mean(x_b)
        se = np.sqrt(np.var(x_a, ddof=1) / len(x_a) + np.var(x_b, ddof=1) / len(x_b))
        est_survey.append(diff)
        covered += (diff - 1.96 * se <= TRUE_IMPROVEMENT <= diff + 1.96 * se)
        n_resp.append(len(x_a))

        # The comparison: the same number of respondents, but picked at random from everyone
        r_b = b_scores[sim_rng.choice(n, size=len(x_b), replace=False)]
        r_a = a_scores[sim_rng.choice(n, size=len(x_a), replace=False)]
        est_random.append(np.mean(r_a) - np.mean(r_b))

    est_survey, est_random = np.array(est_survey), np.array(est_random)
    print(f"{n:>8,} {int(np.mean(n_resp)):>12,} | {est_survey.mean():>11.3f} {est_survey.std(ddof=1):>6.3f} "
          f"{est_survey.mean() - TRUE_IMPROVEMENT:>+7.3f} {covered / n_reps:>9.0%} |"
          f" {est_random.mean():>11.3f} {est_random.std(ddof=1):>6.3f} {est_random.mean() - TRUE_IMPROVEMENT:>+7.3f}")

True improvement: 0.100

   users  respondents | survey est.     SD   error CI covers | random est.     SD   error
--------------------------------------------------------------------------------------------
   5,000          769 |       0.697  0.035  +0.597        0% |       0.091  0.035  -0.009
  20,000        3,067 |       0.709  0.014  +0.609        0% |       0.095  0.022  -0.005
 100,000       15,277 |       0.709  0.007  +0.609        0% |       0.096  0.011  -0.004
 500,000       76,580 |       0.709  0.003  +0.609        0% |       0.099  0.004  -0.001


### Step 3.2: Power, before the fact

**Sample size for a true improvement of 0.1 points.** For 80% power at alpha = 0.05, a 0.1-point improvement is d = 0.12, which takes about 1,044 respondents per group if using the survey's SD (0.815), or about 1,236 per group if using the true spread among all users (0.887). The two differ because selection thinned out the tail. The post-redesign respondents have an SD of only 0.68 against 0.88 before.

**What is detectable at the sample the PM can actually get.** With 7,597 respondents after and 13,768 before, the smallest effect detectable at 80% power is d = 0.040, about 0.033 to 0.035 points. That is a third of the true improvement, so the existing survey was never underpowered for 0.1.

**Would a larger sample make the estimate better?** No, it would make it more precise and no less wrong. The simulation repeats the follow-up survey at four sizes under the Phase 1 response mechanism, and compares it with a random sample of the same number of respondents.

**Why power does not help here.** Power tells how likely the test is to detect a difference in the quantity the survey estimates. That quantity is the gap between respondents (about 0.71), not the gap between all users (0.10). So a power calculation done "for 0.1" gives a number that does not apply to what the survey measures.

**What would help** Changing who answers rather than how many answer. Following up with nonrespondents, weighting by the covariates that drive responses (engagement and support tickets), or modelling the response mechanism. The random-sample column is an idealisation, since it assumes we can pick users at random and make them answer, but a real survey cannot.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* Chat GPT-6

*Version 1 prompt:*  

*Reply:*

*Version 2 prompt:*

*Reply:*

###**AI Transcript, Model: Chat GPT, 10/26/2026**

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

In [58]:
#AI Transcript
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)

covariates = ["engagement_minutes", "had_support_ticket"]
outcome = "satisfaction"
response = "responded"

# Fit the response-probability model
response_model = LogisticRegression(max_iter=1000)
response_model.fit(df[covariates], df[response])

# Estimate response probabilities for respondents
respondents = df[df[response] == 1].copy()
respondents["p_response"] = response_model.predict_proba(
    respondents[covariates]
)[:, 1]

# Inverse-probability weighted population mean
respondents["weight"] = 1 / respondents["p_response"]

weighted_mean = np.average(
    respondents[outcome],
    weights=respondents["weight"]
)

# Nonparametric bootstrap: resample population units and refit model
n_boot = 2000
bootstrap_estimates = []

for _ in range(n_boot):
    boot = df.iloc[
        rng.integers(0, len(df), size=len(df))
    ].copy()

    # Both response classes must be present
    if boot[response].nunique() < 2:
        continue

    model = LogisticRegression(max_iter=1000)
    model.fit(boot[covariates], boot[response])

    boot_respondents = boot[boot[response] == 1].copy()

    if boot_respondents.empty:
        continue

    p = model.predict_proba(
        boot_respondents[covariates]
    )[:, 1]

    # Avoid numerical division by zero
    p = np.clip(p, 1e-6, 1)

    estimate = np.average(
        boot_respondents[outcome],
        weights=1 / p
    )
    bootstrap_estimates.append(estimate)

ci_lower, ci_upper = np.percentile(
    bootstrap_estimates, [2.5, 97.5]
)

print(f"IPW estimate: {weighted_mean:.3f}")
print(f"Bootstrap 95% CI: ({ci_lower:.3f}, {ci_upper:.3f})")

# Available only because this is a simulation:
true_mean = df[outcome].mean()
observed_mean = respondents[outcome].mean()

print(f"True population mean: {true_mean:.3f}")
print(f"Unweighted respondent mean: {observed_mean:.3f}")


KeyError: 'responded'

###**AI Prompt Revision**

A main issue is that the code couldn't run on the data. The prompt says “reuse my variable names” but never lists them, so the AI invented a DataFrame called df with a responded column instead of using pop_before and pop_after, and the response indicators resp_before and resp_after are separate boolean arrays, not columns. The mechanism is also vague. “Probability of responding increases with satisfaction” doesn’t give the formula, so the AI can’t reason about bias without knowing that response depends on the latent s, with different coefficients per period.

To fix these issues I made the revised prompt more specific. I gave the AI the exact names and I wrote out the response formula with its coefficients and said it depends on the latent s, and I defined the truth as 3.30 and 3.40, so the AI could reason about why the correction fails. I also replaced the leading “answer yes or no” with a request for numbers, and added a format section so the answer comes in a predictable order.

###**Version 2 prompt, Chat GPT-6, 10/26/2026:**

[Prep]
Act as a survey statistician who works on non-response adjustment. Do not claim a
correction works unless the simulation shows it. If a result contradicts what you
expected, say so.

[Data: use these exact names]
- pop_before, pop_after: pandas DataFrames, 50,000 users each, with columns
  engagement_minutes, engagement_z, had_support_ticket, s (latent satisfaction), and
  satisfaction (the reported 1-5 score: s rounded and clipped).
- resp_before, resp_after: boolean NumPy arrays (not columns) marking who answered.
- TRUE_MEAN_BEFORE = 3.300 and TRUE_MEAN_AFTER = 3.400 are the truth (the mean of s).
- The survey only observes satisfaction for respondents.

[Mechanism]
Response is generated by P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
- 0.25*had_support_ticket), with a = -1.0, b = 0.45 before and a = -2.75, b = 2.0 after.
It depends on the latent s, not on the reported score.

[Request]
Write Python that estimates each period's mean with inverse-probability weighting:
fit a logistic regression of response on engagement_z and had_support_ticket, weight each
respondent by 1 / fitted probability, and compute the weighted mean (weights normalised
to sum to 1). Return a table with the unweighted respondent mean, the IPW estimate, and
a 95% bootstrap interval for each period.

[Constraints]
- Use only numpy, pandas and statsmodels (Logit, no regularisation).
- Bootstrap by resampling users and refitting the response model, B = 500.
- Report the largest weight in each period.
- Use the generator rng = np.random.default_rng(42) for all random draws.

[Validation: run both]
1. Positive control: change the mechanism so response depends only on the two covariates
   (b = 0). Show whether IPW recovers the truth.
2. The real mechanism above: report the share of the respondent-mean bias that IPW removes.

[Mechanism Check]
Name the assumptions that make IPW valid (ignorability / missing at random given the
covariates, and positivity). Then explain, using this simulation's equation, why
response depending on s violates the first one even though s is partly a function of the
covariates. Also say what the bootstrap interval does and does not measure.

[Format]
1. Assumptions (short). 2. One code block. 3. Results table. 4. Evaluation, with numbers.

In [59]:
#Version Two AI Code
import numpy as np
import pandas as pd
import statsmodels.api as sm

rng = np.random.default_rng(42)
B = 500

covariates = ["engagement_z", "had_support_ticket"]


def estimate_period(pop, resp, period):
    X = sm.add_constant(
        pop[covariates].astype(float),
        has_constant="add"
    )
    y = pop["satisfaction"].to_numpy(dtype=float)
    resp = np.asarray(resp, dtype=bool)

    def fit_ipw(indices):
        Xb = X.iloc[indices]
        yb = y[indices]
        rb = resp[indices]

        model = sm.Logit(
            rb.astype(int), Xb
        ).fit(disp=False)

        respondent_mask = rb
        p = np.asarray(model.predict(Xb))[respondent_mask]
        weights = 1 / p
        weights_normalized = weights / weights.sum()

        estimate = np.sum(
            weights_normalized * yb[respondent_mask]
        )
        return estimate, weights.max()

    n = len(pop)
    estimate, largest_weight = fit_ipw(np.arange(n))
    unweighted_mean = y[resp].mean()

    bootstrap_estimates = []
    attempts = 0

    while len(bootstrap_estimates) < B and attempts < 5 * B:
        attempts += 1
        indices = rng.integers(0, n, size=n)

        try:
            boot_estimate, _ = fit_ipw(indices)
            bootstrap_estimates.append(boot_estimate)
        except Exception:
            continue

    if len(bootstrap_estimates) < B:
        raise RuntimeError(
            f"Could not obtain {B} valid bootstrap replicates for {period}."
        )

    lower, upper = np.percentile(
        bootstrap_estimates, [2.5, 97.5]
    )

    return {
        "Period": period,
        "Unweighted respondent mean": unweighted_mean,
        "IPW estimate": estimate,
        "95% CI lower": lower,
        "95% CI upper": upper,
        "Largest weight": largest_weight,
    }


results = pd.DataFrame([
    estimate_period(pop_before, resp_before, "Before"),
    estimate_period(pop_after, resp_after, "After"),
])

print(results.round(4).to_string(index=False))


Period  Unweighted respondent mean  IPW estimate  95% CI lower  95% CI upper  Largest weight
Before                       3.600         3.464         3.449         3.477           8.117
 After                       4.304         4.143         4.122         4.164          44.096


### Task 4.2: The slide (5 pts)

Among survey respondents, satisfaction increased from 3.60 to 4.30, a rise of 0.70 points (95% CI 0.683, 0.725). However, this result should not be interpreted as evidence that users overall are substantially more satisfied. The response rate declined from 27.5% to 15.2%, meaning that the composition of respondents may have changed between periods. If the survey increasingly captures feedback from highly satisfied and engaged users, the observed improvement may overstate the actual change in overall satisfaction. Adjusting for engagement and support history using inverse-probability weighting reduces the estimated increase to 0.68 points. However, this adjustment cannot account for differences in satisfaction that influence response behavior independently of the measured covariates. The confidence interval reflects sampling uncertainty, not bias from non-response. A simulation of this response pattern demonstrated that an improvement of 0.10 points could produce a similarly observed increase, highlighting the limitations of the current findings. The results support a rise in respondent satisfaction, but the magnitude of improvement across the user population is uncertain. To strengthen future evaluations, random-sample users who complete onboarding should be selected, and follow-ups should be conducted with nonrespondents to improve response rates and have a more representative assessment of satisfaction.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.